# Custom Graph IV

Objectives:
1. Implement conditional logic to route the flow of data to different nodes
2. Use START and END Nodes to manage entry and exit points explicitly
3. Design multiple nodes to perform different operations
4. Create two router nodes to handle decision-making and control graph flow


In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [10]:
#State schema
class AgentState(TypedDict):
    #Pass in two numbers and an operation
    number1: int
    operation1: str
    number2: int
    operation2: str
    number3: int
    number4: int
    finalNumber: int
    finalNumber2: int

In [11]:
#First set of nodes for router 1
def adder_1(state: AgentState) -> AgentState:
    """This node adds number1 and number2"""
    state["finalNumber"] = state["number1"] + state["number2"]
    return state

def subtractor_1(state: AgentState) -> AgentState:
    """This node subtracts number2 from number1"""
    state["finalNumber"] = state["number1"] - state["number2"]
    return state

# Router 1 - decides between addition or subtraction for first operation
def router_1_decision(state: AgentState) -> str:
    """Router 1: decides based on operation1"""
    if state["operation1"] == "+":
        return "addition_1"
    elif state["operation1"] == "-":
        return "subtraction_1"

#Second set of nodes for router 2  
def adder_2(state: AgentState) -> AgentState:
    """This node adds number3 and number4"""
    state["finalNumber2"] = state["number3"] + state["number4"]
    return state

def subtractor_2(state: AgentState) -> AgentState:
    """This node subtracts number4 from number3"""
    state["finalNumber2"] = state["number3"] - state["number4"]
    return state

# Router 2 - decides between addition or subtraction for second operation
def router_2_decision(state: AgentState) -> str:
    """Router 2: decides based on operation2"""
    if state["operation2"] == "+":
        return "addition_2"
    elif state["operation2"] == "-":
        return "subtraction_2"

In [12]:
#Build the graph with two routers and four conditional nodes
graph = StateGraph(AgentState)

# Add all four operation nodes
graph.add_node("add_node_1", adder_1)
graph.add_node("subtract_node_1", subtractor_1)
graph.add_node("add_node_2", adder_2)
graph.add_node("subtract_node_2", subtractor_2)

# Add router nodes (these are just passthrough nodes for conditional logic)
graph.add_node("router_1", lambda state: state)
graph.add_node("router_2", lambda state: state)

# Set the entry point to router_1
graph.add_edge(START, "router_1")

# Router 1 conditional edges - decides between first two operations
graph.add_conditional_edges(
    "router_1",
    router_1_decision,
    {
        "addition_1": "add_node_1",
        "subtraction_1": "subtract_node_1"
    }
)

# After first operation, go to router_2
graph.add_edge("add_node_1", "router_2")
graph.add_edge("subtract_node_1", "router_2")

# Router 2 conditional edges - decides between second two operations
graph.add_conditional_edges(
    "router_2", 
    router_2_decision,
    {
        "addition_2": "add_node_2",
        "subtraction_2": "subtract_node_2"
    }
)

# Both final operations end the graph
graph.add_edge("add_node_2", END)
graph.add_edge("subtract_node_2", END)

app = graph.compile()

In [13]:
# Test the graph with two routers
test_input = {
    "number1": 10,
    "operation1": "+",  # Router 1 will choose addition
    "number2": 5,
    "operation2": "-",  # Router 2 will choose subtraction  
    "number3": 20,
    "number4": 8,
    "finalNumber": 0,
    "finalNumber2": 0
}

result = app.invoke(test_input)
print("Result:", result)
print(f"First operation: {result['number1']} {result['operation1']} {result['number2']} = {result['finalNumber']}")
print(f"Second operation: {result['number3']} {result['operation2']} {result['number4']} = {result['finalNumber2']}")

Result: {'number1': 10, 'operation1': '+', 'number2': 5, 'operation2': '-', 'number3': 20, 'number4': 8, 'finalNumber': 15, 'finalNumber2': 12}
First operation: 10 + 5 = 15
Second operation: 20 - 8 = 12
